# EyeAI Product Backend Core V1 — Kaggle Smoke Test

This notebook does not train the model. It verifies the complete protected product workflow:

`bootstrap → login → patient → eye visit → Run 09 analysis → timeline → alerts → PDF report`.


## 1. Paths and smoke-test controls

The database and generated artifacts are isolated under `/kaggle/working` and may be reset safely.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import shutil
import subprocess
import sys

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")
API_CONFIG = REPO_DIR / "configs/api/product_backend_v1.yaml"
SMOKE_ROOT = Path("/kaggle/working/eyeai_product_backend_smoke")
RESET_SMOKE_DATABASE = True

if RESET_SMOKE_DATABASE and SMOKE_ROOT.exists():
    shutil.rmtree(SMOKE_ROOT)
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)

print("Smoke-test workspace:", SMOKE_ROOT)


## 2. Clone and install the current project

This cell retrieves all Product Backend Core files from GitHub and installs the local package.


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

repo_src = str(REPO_DIR / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
print("Repository and package are ready.")


## 3. Discover the frozen Run 09 + TTA model package and prepared data

Only the exported Model Package V1 is required for inference. Old training checkpoints are not needed.


In [ ]:
def discover_model_package() -> Path:
    candidates = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob("model.pth")
        if (path.parent / "model_config.yaml").is_file()
        and (path.parent / "threshold.json").is_file()
        and (path.parent / "version.json").is_file()
    })
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one complete Model Package V1, found: {candidates}")
    return candidates[0]


def discover_dataset_root() -> Path:
    candidates = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob("dataset_summary.json")
        if (path.parent / "manifests/hyamd_val.csv").is_file()
    })
    if not candidates:
        raise FileNotFoundError("Attach hymd-armd-dataset for the smoke test.")
    return candidates[0]

MODEL_PACKAGE_DIR = discover_model_package()
DATASET_ROOT = discover_dataset_root()

print("Model package:", MODEL_PACKAGE_DIR)
print("Prepared dataset:", DATASET_ROOT)
print(f"model.pth size: {(MODEL_PACKAGE_DIR / 'model.pth').stat().st_size / (1024**3):.3f} GB")


## 4. Configure an isolated database, report directory, and JWT secret

The secret below is used only for this disposable smoke test. Production must use `EYEAI_JWT_SECRET` from a secure secret store.


In [ ]:
os.environ["EYEAI_DATABASE_URL"] = f"sqlite:///{SMOKE_ROOT / 'eyeai.db'}"
os.environ["EYEAI_EXPLANATION_OUTPUT_DIR"] = str(SMOKE_ROOT / "explanations")
os.environ["EYEAI_REPORTS_OUTPUT_DIR"] = str(SMOKE_ROOT / "reports")
os.environ["EYEAI_JWT_SECRET"] = "kaggle-smoke-test-secret-change-before-production-2026"

from eyeai.api.config import ApiSettings
from eyeai.api.main import create_app

settings = ApiSettings.from_yaml(
    API_CONFIG,
    model_package_override=MODEL_PACKAGE_DIR,
    device_override="cuda" if __import__("torch").cuda.is_available() else "cpu",
)
print("Database:", settings.database_url)
print("Device:", settings.device)


## 5. Start the API in-process and create the first administrator

The bootstrap endpoint is usable only while the users table is empty.


In [ ]:
from fastapi.testclient import TestClient

app = create_app(settings)
client_context = TestClient(app)
client = client_context.__enter__()

health = client.get("/health")
health.raise_for_status()
print("Health:", json.dumps(health.json(), indent=2))

bootstrap = client.post(
    "/api/v1/auth/bootstrap",
    json={
        "email": "doctor.smoke@eyeai.local",
        "full_name": "EyeAI Smoke Doctor",
        "password": "EyeAI-Smoke-Password-2026",
    },
)
bootstrap.raise_for_status()

login = client.post(
    "/api/v1/auth/login",
    json={
        "email": "doctor.smoke@eyeai.local",
        "password": "EyeAI-Smoke-Password-2026",
    },
)
login.raise_for_status()
TOKEN = login.json()["access_token"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print("Authenticated user:", client.get("/api/v1/auth/me", headers=HEADERS).json())


## 6. Create a patient and a right-eye visit

This validates patient persistence, eye laterality, and protected endpoints.


In [ ]:
patient_response = client.post(
    "/api/v1/patients",
    headers=HEADERS,
    json={
        "medical_record_number": "SMOKE-AMD-001",
        "first_name": "Demo",
        "last_name": "Patient",
        "sex": "unspecified",
    },
)
patient_response.raise_for_status()
PATIENT = patient_response.json()

visit_response = client.post(
    f"/api/v1/patients/{PATIENT['id']}/visits",
    headers=HEADERS,
    json={"eye": "right", "notes": "Kaggle Product Backend V1 smoke test."},
)
visit_response.raise_for_status()
VISIT = visit_response.json()

print("Patient:", json.dumps(PATIENT, indent=2))
print("Visit:", json.dumps(VISIT, indent=2))


## 7. Analyze a real AMD image and persist its explanation

The result is stored in the database together with TTA, quality, and explanation metadata.


In [ ]:
import pandas as pd
from PIL import Image

validation = pd.read_csv(DATASET_ROOT / "manifests/hyamd_val.csv", dtype={"image_id": str})
row = validation[validation["binary_label"] == 1].sample(1, random_state=42).iloc[0]
image_path = DATASET_ROOT / row["relative_image_path"]

with image_path.open("rb") as handle:
    analyze_response = client.post(
        f"/api/v1/visits/{VISIT['id']}/analyze?explanation=true",
        headers=HEADERS,
        files={"file": (image_path.name, handle.read(), "image/jpeg")},
    )
analyze_response.raise_for_status()
PREDICTION = analyze_response.json()

print(json.dumps(PREDICTION, indent=2, ensure_ascii=False))
display(Image.open(image_path).convert("RGB"))


## 8. Validate notes, timeline, alerts, dashboard, and PDF report

The timeline uses neutral model-score language and does not claim clinical progression.


In [ ]:
note = client.post(
    f"/api/v1/visits/{VISIT['id']}/notes",
    headers=HEADERS,
    json={"text": "Smoke-test note: review the model output clinically."},
)
note.raise_for_status()

timeline = client.get(
    f"/api/v1/patients/{PATIENT['id']}/timeline?eye=right",
    headers=HEADERS,
)
timeline.raise_for_status()

alerts = client.get("/api/v1/alerts", headers=HEADERS)
alerts.raise_for_status()

dashboard = client.get("/api/v1/dashboard", headers=HEADERS)
dashboard.raise_for_status()

report = client.post(f"/api/v1/visits/{VISIT['id']}/reports", headers=HEADERS)
report.raise_for_status()
report_payload = report.json()
report_download = client.get(report_payload["download_url"], headers=HEADERS)
report_download.raise_for_status()
report_path = SMOKE_ROOT / "downloaded_report.pdf"
report_path.write_bytes(report_download.content)

print("Timeline:")
print(json.dumps(timeline.json(), indent=2, ensure_ascii=False))
print("
Alerts:")
print(json.dumps(alerts.json(), indent=2, ensure_ascii=False))
print("
Dashboard:")
print(json.dumps(dashboard.json(), indent=2, ensure_ascii=False))
print("
PDF report:", report_path, report_path.stat().st_size, "bytes")

assert timeline.json()[0]["trend"] == "first_measurement"
assert report_path.stat().st_size > 1000
print("
Product Backend Core V1 smoke test passed.")


## 9. Close the in-process client

The generated database, explanations, and report remain under `/kaggle/working/eyeai_product_backend_smoke` for inspection.


In [ ]:
client_context.__exit__(None, None, None)
print("Client closed.")
